# 리포트 24 — σ = A(f)·B₁·B₂ 에서 A(f) 의 기울기만 측정에서 받고, 레벨과 각패턴은 우리 SBR+PO 커널(B) 출력이다

> ### 한 일
> **게재된 표준트랙 논문의 분해를 그대로 쓰고, 세 인자 중 주파수 기울기 하나만 공개 측정에 정렬했다.**

### 결과
1. 기하에서 나온 우리 밴드 기울기는 기체 5 종 [^1] 범위로 -0.084 [^2] (mavic4pro [^3])~2.004 dB/GHz [^4] (phantom4 [^5]) 이고, 측정은 0.210 [^6] (Das) 와 0.315 dB/GHz [^7] (Yuan) 다.
2. 생산 기준은 `slope_only [^8]` 이고, 밴드별로 옮기는 양은 +2.70 [^9] (phantom4 [^10] @ LTE 1.843 GHz [^11]) ~ -2.41 dB [^12] (s1000plus [^13] @ WiFi 5.21 GHz [^14]) 다.
3. ⚠ 정렬 후 7 기종 [^15] 기울기가 모두 0.210 dB/GHz [^16] 에 서고 기종 간 산포가 1.9e-15 dB/GHz [^17] 인 것은 관측이 아니라 **정의**다 — `slope_only [^8]` 은 밴드 비가중 평균 레벨을 축으로 회전만 시키므로 정렬 후 기울기가 앵커 기울기 0.210 dB/GHz [^6] (Das) 와 항등적으로 같아진다 [^18]. 이 산포는 부동소수 반올림 자리다.
4. ⚠ 레벨이동 절대 최대 0.00 dB [^19] 도 같은 항등식의 구현 검산이다 — 정의상 0 인 것은 세 밴드 **평균** Δ 이고, 밴드별 Δ 는 위 2번 범위다. 함께 실린 정규화 각패턴 변화 1.9e-15 dB [^20] 는 `src/sigma_anchor.py` 가 생산 모드가 아니라 `level_and_slope_L2` 에서, LTE 1.843 GHz [^21] 한 밴드의 el=0 격자 120 방위에서만 잰 값이다.
5. 레벨까지 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택 하나가 기체당 최대 9.50 dB [^22] 를 정한다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 분해 | σ = A(f)·B₁(φ,θ)·B₂ — 게재된 표준트랙 논문의 분해를 그대로 쓴다(Zhang, IEEE JSAC 44:702, 2026 — 측정 적합 모델) |
| 무엇을 받나 | A(f) 의 **기울기만** 측정에서 받는다. A(f) 의 레벨과 B₁ 은 우리 SBR+PO 커널(B) 출력이다 |
| 팔 표기 | **S** = Sionna PathSolver · **B** = 우리 SBR+PO 커널(광선 가림 + 셸 투과 + PO 면적분) · **P** = 순수 PO 대조군(PEC \|Γ\|=1) — 이 편의 σ 는 전부 B 팔이다 |
| 왜 기울기만 | 레벨까지 맞추려면 크기전이 법칙을 하나 골라야 하고, 측정이 그 대가 없이 제약하는 양은 기울기뿐이다 |
| 모드 구현 | `src/sigma_anchor.py` 가 재보정 모드 셋을 제공한다 — 생산은 `slope_only` 다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python benchmark/rcs_anchor.py
PYTHONPATH=src python src/build_part05_anchor.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json`, `outputs/rcs_anchor.json`, `outputs/sigma_anchor.json` |
| 소요 | 약 3분 (GPU 0장 — 이미 낸 σ 격자에 적합을 다시 건다) |
| 비고 | PO 면적분은 f² 로 커지는 정반사항을 담는다 — 그 기울기를 측정과 맞춘다 |

---

## 세 인자로 갈라 놓고 하나만 건드린다

게재된 표준트랙 논문의 분해를 그대로 쓴다(Zhang, IEEE JSAC 44:702, 2026 — 측정 적합 모델): **σ = A(f)·B₁(φ,θ)·B₂**. A(f) 는 주파수 의존성, B₁ 은 자세에 따른 모양, B₂ 는 자세 요동의 분포족이다.

**A(f) 의 기울기는 측정에서, A(f) 의 레벨과 B₁ 은 우리 SBR+PO 커널(B)에서 온다.** PO 면적분은 f² 로 커지는 정반사항을 담으므로 기울기가 기하에서 나오고, 그 하나를 측정에 맞춘다.

팔 표기는 **S** = Sionna PathSolver · **B** = 우리 SBR+PO 커널(광선 가림 + 셸 투과 + PO 면적분) · **P** = 순수 PO 대조군(PEC |Γ|=1) 다. B 팔의 신원은 원장이 적어 둔다 — 엔진 «SBR (Mitsuba/OptiX first-hit) + PO surface integral — src/rcs_sbr.rcs_sbr_batch [^23]», 호출자 «benchmark/rcs_anchor.raw_sigma_az (다른 기체와 동일 경로·동일 인자 규약) [^24]» 로 앵커 사슬의 σ 는 전부 이 경로를 지난다.

## 우리 기울기와 측정 기울기

기하에서 나온 우리 밴드 기울기는 기체 5 종 [^1] 범위로 -0.084 [^2] (mavic4pro [^3])~2.004 dB/GHz [^4] (phantom4 [^5]) 이고, 측정은 0.210 [^6] (Das, IEEE WCL 2026) 와 0.315 dB/GHz [^7] (Yuan, EuCAP 2025) 다.

⚠ **두 수는 창이 다르다** — 우리 값은 생산 세 밴드(1.843 [^21]~5.21 GHz [^25])에서 잰 기울기이고 측정 두 값은 1.8 [^26]~18.2 GHz [^27] 전대역 적합이다. 같은 기체를 같은 커널로 돌려도 창을 바꾸면 기울기가 달라진다 — 우리 Phantom 3 v2 메쉬에서 저대역 창 1.018 [^28] 대 전대역 0.519 dB/GHz [^29] 다.

⚠ **이 범위의 모집단은 기체 5 종 [^1]** 이다 — 모드 표와 정렬 후 산포가 드는 7 기종 [^15] 중 2 대 [^30] (typhoonh480, x500v2 [^31])는 σ 앵커 원장에만 있고 밴드 기울기 원장 밖이다. 두 수를 나란히 읽을 때 이 차이를 함께 읽는다.

## 앵커는 각 기체의 기울기를 어디로 옮기나

![report02_f6_band_slope](../outputs/figures/report02_f6_band_slope.png)

**그림 1.** 측정 앵커는 각 기체의 밴드 기울기를 어디로 옮기는가?

## 모드 선택 — 무엇을 옮기고 무엇을 대가로 내는가

| 모드 | 무엇을 옮기나 | 평균 레벨이동 [dB] | 대가 |
|---|---|---|---|
| slope_only (기본) | 주파수 의존성 A(f) 의 기울기만 | 0.00 | 없음 — 크기 가정 0개 |
| level_and_slope_L2 | 레벨 + 기울기 | -0.82 ~ +7.42 | 크기전이 L² 가정 (σ ∝ 투영면적) |
| level_and_slope_L4 | 레벨 + 기울기 | +1.93 ~ +16.92 | 크기전이 L⁴ 가정 (σ ∝ A²/λ²) · L² 와 최대 9.50 dB 차 (DJI S1000+) |

출처 [^32]

생산 기준은 `slope_only [^8]` 이고, 밴드별로 옮기는 양은 +2.70 [^9] (phantom4 [^10] @ LTE 1.843 GHz [^11]) ~ -2.41 dB [^12] (s1000plus [^13] @ WiFi 5.21 GHz [^14]) 다. 레벨이동 열은 기체 7 종 [^15] 의 세 밴드 평균 Δ 다.

⚠ **`slope_only [^8]` 행의 평균 레벨이동 «0.00 [^33]» 은 잰 값이 아니라 모드의 정의다** — 원장이 그렇게 적어 둔다 [^18]. 이 모드는 목표를 `target = μ_anchor + (mean(μ_our) − mean(μ_anchor))` 로 잡으므로(`src/sigma_anchor.py` 의 `relevel()`) 세 밴드 평균 Δ 가 항등적으로 0 이고, 바로 위의 +2.70 [^9] ~ -2.41 dB [^12] 가 그 0 을 이루는 밴드별 성분이다.

## 왜 기울기만 받나

레벨까지 앵커에 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택이 기체에 따라 최대 9.50 dB [^22] 를 정한다. 측정이 그 대가 없이 제약하는 양은 기울기뿐이므로 기울기만 받는다 [^34].

그 선택을 측정으로 닫는 자리가 [편 74 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](74_sim-vs-meas.ipynb) 이고, 절대 레벨을 처음으로 앵커하는 것은 [편 70 «구가 σ 를 절대량으로 만들고»](70_calibration-sphere.ipynb) 의 교정구다.

## 세 인자, 각각의 출처

| 인자 | 무엇 | 어디서 | 이 편의 근거 |
|---|---|---|---|
| A(f) 기울기 | 주파수 의존성 | **측정**(Das) | μ 기울기 0.21 dB/GHz [^6] |
| A(f) 레벨 | 절대 레벨 | **우리 SBR+PO 커널(B) 출력** | 설계 — `slope_only [^8]` 이 밴드 평균 레벨을 보존한다. 레벨이동 절대 최대 0.00 dB [^19] 는 그 항등식의 구현 검산 |
| B₁(φ,θ) | 자세에 따른 모양 | **기하**(B — 광선 가림 + 셸 투과 + PO) | 설계 — 보정이 밴드별 스칼라 곱이다. 정규화 패턴 변화 1.9e-15 dB [^20] 는 `level_and_slope_L2` · LTE 1.843 GHz [^21] 한 밴드에서 잰 검산 |
| B₂ | 자세 요동의 분포족 | 기하(B) | 문헌 적합 RMSE 2.41 dB [^35] 가 기준선 |

⚠ **넷째 열은 셋째 열의 증거가 아니다.** A(f) 레벨·B₁ 두 행의 숫자는 재보정이 레벨과 각패턴을 건드리지 않도록 짜였다는 설계의 구현 검산이고, 그 두 행의 «어디서» 도 잰 결과가 아니라 이 편이 고른 설계다. 원장도 그렇게 적는다 — 측정에서 받는 것은 «A(f) 의 주파수 기울기 [dB/GHz] [^36]» 하나이고, «절대 레벨 A(f)|_mean · 자세 패턴 B₁(φ,θ) · 잔차 분포 B₂ [^37]» 는 우리 커널에서 온다.

## 이 표가 서 있는 사슬 세대

⚠ **생산 앵커 원장과 위 문단의 우리 기울기는 σ 사슬의 서로 다른 세대다.** 생산 원장은 2026-07-30 07:16:08 [^38] 판 `rcs_anchor.json` 위에 서 있고, 위 문단은 디스크의 현재 판(2026-08-03 05:49:19 [^39])에서 다시 적합한 값이다. 겹치는 5 기체 [^1] 에서 밴드 기울기가 최대 1.396 dB/GHz [^40] 갈린다(mavic4pro [^41]).

⚠ **그리고 두 세대 모두 2026-08-04 [^42] 형상 정정 전 메쉬 위에 서 있다** — 이 축은 위 문단의 세대 축과 별개다. 기울기를 대조한 5 기체 [^1] 중 Matrice 4E 가 그 정정을 받았다 [^43].

⚠ 셋째 축 — 두 세대 모두 2026-08-07 10:58:22 [^44] Γ(θ) 각도 모양(기본 켬) 이전 커널의 산출이기도 하다 — 전체 드론 σ 이동 +0.08 [^45] ~ +0.10 dB [^46] 다. 앵커를 갈아끼우려면 σ 사슬 전체를 형상 정정 + Γ(θ) 한 세대로 맞춰야 하고, 그것은 [부 10 «검출 결과»](../README.md#부-10-검출-결과) 전체가 먹는 값이다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 앵커 사슬(rcs_anchor → sigma_anchor)을 현재 메쉬·Γ(θ) 켠 커널로 다시 돌린다 | 밴드 기울기와 재보정 원장이 현재 기하 위에 선다 — 재생성 평균이 기체마다 -5.70 [^47] ~ +3.18 dB [^48] 로 갈린다 | `benchmark/rcs_anchor.py` → `src/sigma_anchor.py` |
| R90 사슬에 `modes.slope_only.delta_db` 를 적용한다 | 앵커 후 파형 순위가 확정된다 — 지금 적용하면 2 기체 [^49] 의 순위가 바뀐다 | `src/experiment_freespace_sigma.py` → [편 59 «레벨을 맞추려면 크기전이 법칙을 골라야 하므로…»](59_slope-anchor.ipynb) |
| Matrice 4E · Mini 5 Pro 의 상대 레벨을 실측한다 | 크기전이 지수가 직접 고정되어 앵커 원장의 최대 항이 닫힌다 | [편 74 «캠페인이 결판내는 양은 절대값이 아니라 순위다»](74_sim-vs-meas.ipynb) |
| 모서리 회절항(PTD)을 켜고 밴드 기울기를 다시 적합한다 | 면적분 밖의 항이 주파수 축을 얼마나 옮기는지가 확정된다 | `benchmark/rcs_anchor.py --ptd` |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 49개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report02_derived.json` | `anchor.n_slope_crosschecked` | 5 |
| [^2] | `outputs/report02_derived.json` | `band_slope.ours_min` | -0.08445 |
| [^3] | `outputs/report02_derived.json` | `band_slope.ours_min_drone` | mavic4pro |
| [^4] | `outputs/report02_derived.json` | `band_slope.ours_max` | 2.004 |
| [^5] | `outputs/report02_derived.json` | `band_slope.ours_max_drone` | phantom4 |
| [^6] | `outputs/rcs_anchor.json` | `literature.mu_eps.das_phantom3_mono.mu_a` | 0.21 |
| [^7] | `outputs/rcs_anchor.json` | `literature.mu_eps.yuan_phantom3_azplane.mu_a` | 0.315 |
| [^8] | `outputs/report02_derived.json` | `anchor_modes.production_mode` | slope_only |
| [^9] | `outputs/report02_derived.json` | `anchor.correction_max_db` | 2.698 |
| [^10] | `outputs/report02_derived.json` | `anchor.correction_max_drone` | phantom4 |
| [^11] | `outputs/report02_derived.json` | `anchor.correction_max_band` | LTE 1.843 GHz |
| [^12] | `outputs/report02_derived.json` | `anchor.correction_min_db` | -2.41 |
| [^13] | `outputs/report02_derived.json` | `anchor.correction_min_drone` | s1000plus |
| [^14] | `outputs/report02_derived.json` | `anchor.correction_min_band` | WiFi 5.21 GHz |
| [^15] | `outputs/report02_derived.json` | `anchor_modes.n_airframes` | 7 |
| [^16] | `outputs/report02_derived.json` | `anchor.slope_after_db_per_ghz` | 0.21 |
| [^17] | `outputs/report02_derived.json` | `anchor.slope_after_spread_db_per_ghz` | 1.887e-15 |
| [^18] | `outputs/report02_derived.json` | `anchor_modes.definition` | 평균 레벨이동 = 세 밴드 delta_db 의 산술평균 [dB], 기체 7종에 대한 최소~최대. s… |
| [^19] | `outputs/report02_derived.json` | `anchor_modes.level_shift_abs_max_db` | 4.737e-15 |
| [^20] | `outputs/report02_derived.json` | `anchor.shape_invariance_max_abs_db` | 1.929e-15 |
| [^21] | `outputs/report02_derived.json` | `bands_ghz.LTE` | 1.843 |
| [^22] | `outputs/report02_derived.json` | `anchor_modes.size_law_spread_max_db` | 9.501 |
| [^23] | `outputs/p3_ours_v2.json` | `meta.engine` | SBR (Mitsuba/OptiX first-hit) + PO surface integral — s… |
| [^24] | `outputs/p3_ours_v2.json` | `meta.caller` | benchmark/rcs_anchor.raw_sigma_az (다른 기체와 동일 경로·동일 인자 규… |
| [^25] | `outputs/report02_derived.json` | `bands_ghz.WiFi` | 5.21 |
| [^26] | `outputs/p3_validation_v2.json` | `slope.das_published.band[0]` | 1.8 |
| [^27] | `outputs/p3_validation_v2.json` | `slope.das_published.band[1]` | 18.2 |
| [^28] | `outputs/p3_validation_v2.json` | `slope.subband.ours.1.8-6.0 GHz.a` | 1.018 |
| [^29] | `outputs/p3_validation_v2.json` | `slope.ours_el0_full_band.a` | 0.5188 |
| [^30] | `outputs/report02_derived.json` | `anchor.n_in_sigma_anchor_only` | 2 |
| [^31] | `outputs/report02_derived.json` | `anchor.in_sigma_anchor_only` | typhoonh480, x500v2 |
| [^32] | `outputs/report02_derived.json` | `anchor_modes.rows` | (3행 표) |
| [^33] | `outputs/report02_derived.json` | `anchor_modes.rows[0].mean_shift_range_db` | 0.00 |
| [^34] | `outputs/report02_derived.json` | `anchor_modes.why_production` | 레벨까지 앵커에 맞추려면 크기전이 법칙을 하나 골라야 하고, 그 선택이 기체에 따라 최대 9.50… |
| [^35] | `outputs/rcs_anchor.json` | `literature.fit_rmse_db.AAV` | 2.414 |
| [^36] | `outputs/report02_derived.json` | `anchor.from_measurement` | A(f) 의 주파수 기울기 [dB/GHz] |
| [^37] | `outputs/report02_derived.json` | `anchor.from_ours` | 절대 레벨 A(f)\|_mean · 자세 패턴 B₁(φ,θ) · 잔차 분포 B₂ |
| [^38] | `outputs/report02_derived.json` | `anchor.slope_ledger_source_generation` | 2026-07-30 07:16:08 |
| [^39] | `outputs/rcs_anchor.json` | `meta.generated` | 2026-08-03 05:49:19 |
| [^40] | `outputs/report02_derived.json` | `anchor.slope_ledger_gap_max_db_per_ghz` | 1.396 |
| [^41] | `outputs/report02_derived.json` | `anchor.slope_ledger_gap_max_drone` | mavic4pro |
| [^42] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^43] | `outputs/meshfix_attack.json` | `Q6_invalidated_outputs.critical[1]` | (3항목 묶음) |
| [^44] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^45] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.mini5pro.delta_db` | 0.08 |
| [^46] | `outputs/angle_gamma_impact.json` | `whole_drone_sigma_az24_el_-15.matrice4e.delta_db` | 0.1 |
| [^47] | `outputs/s2r_assets_verify.json` | `overstated[0].결과_표[0].mean_delta_db` | -5.7 |
| [^48] | `outputs/s2r_assets_verify.json` | `overstated[0].결과_표[2].mean_delta_db` | 3.18 |
| [^49] | `outputs/report02_derived.json` | `sigma_sens.anchor_not_applied_n_order_changed` | 2 |